# Import

In [53]:
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# Read data

In [54]:
df = pd.read_csv('../data/data.csv')
df

,"Sigma, Mpa",T.K,t.h,Th,C,Cr,Co,Mo,W,Al,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,241.316495,1144.2600,4.5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,NaN,99.500
1,241.316495,1144.2600,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.75,NaN,NaN,NaN,99.250
2,241.316495,1144.2600,68.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.05,99.450
3,241.316495,1144.2600,40.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.20,99.300
4,241.316495,1144.2600,32.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,0.50,NaN,NaN,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,1061.792578,922.0389,1183.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3442,1103.161120,866.4833,25554.7,NaN,0.02,17.0,28.4,3.4,1.9,1.03,...,NaN,6.0,0.05,0.05,NaN,NaN,0.02,NaN,NaN,33.904
3443,241.316495,1199.8170,183.2,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750
3444,241.316495,1199.8170,153.0,NaN,0.18,10.0,15.0,3.0,NaN,5.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.750


In [55]:
mass = '232,03806	12,011	51,9961	58,933194	95,95	183,84	26,9815385	47,867	92,90637	10,81	55,845	88,90584	91,224	180,94788	186,207	101,07	50,9415	140,116	138,90547	32,06	28,085	54,938044	24,305	30,973761998	178,49	107,8682	63,546	208,9804	207,2	192,22	72,63	69,723	58,6934'.replace(',', '.').split()
element = 'Th	C	Cr	Co	Mo	W	Al	Ti	Nb	B	Fe	Y	Zr	Ta	Re	Ru	V	Ce	La	S	Si	Mn	Mg	P	Hf	Ag	Cu	Bi	Pb	Ir	Ge	Ga	Ni'.split()
atom_mass = dict(zip(element, mass))

In [56]:
atom_mass['Ni']

'58.6934'

In [57]:
# for elem in df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list():
#     df[elem] = df[elem] / float(atom_mass[elem])
# df['sum'] = df[df.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns.to_list()].sum(axis=1, skipna=True)

df = df.fillna(0)

df = df.drop(columns=['Sigma, Mpa', 't.h'])

In [58]:
# df_balanced = df.copy()
# cols = df_balanced.drop(columns=['Sigma, Mpa', 'T.K', 't.h']).columns
# df_balanced.loc[:, cols] = df_balanced.loc[:, cols].div(df_balanced['sum'], axis=0)
# df_balanced = df_balanced.drop(columns=['Ni', 'sum'])
# df_balanced.describe()

In [59]:
GROUPS = {
    1: ["Co", "Cr", "Fe", "Mo", "W", "Ta", "Re"],       # Solid solution strengtheners
    2: ["W", "Ta", "Ti", "Mo", "Nb", "Hf"],             # Carbide form MC
    3: ["Cr"],                                          # Carbide form M7C3
    4: ["Cr", "Mo", "W"],                               # Carbide form M23C6
    5: ["Mo", "W", "Nb"],                               # Carbide form M6C
    6: ["C", "N"],                                      # Carbonitrides M(CN)
    7: ["Al", "Ti"],                                    # Forms gamma prime Ni3(Al, Ti)
    8: ["Co"],                                          # Raises solvus temperature of gamma prime
    9: ["Al", "Ti", "Nb"],                              # Hardening precipitates/intermetallics
    10: ["Al", "Cr", "Y", "La", "Ce"],                  # Oxidation resistance
    11: ["La", "Th"],                                   # Hot corrosion resistance
    12: ["Cr", "Co", "Si"],                             # Sulfidation resistance
    13: ["B", "Ta"],                                    # Improves creep properties
    14: ["B"],                                          # Increases rupture strength
    15: ["B", "C", "Zr", "Hf"],                         # Grain-boundary refiners
    16: ["Re", "Ru"],                                   # Retards coarsening
}


In [60]:
target_col = 'T.K'

y = df[target_col]
X = df.copy().drop(columns=[target_col])


In [61]:
y_values = df[target_col].values.reshape(-1, 1).astype("float32")
X_values = X.values.astype("float32")

feature_names = list(X.columns)

print("X shape:", X_values.shape)
print("Number of input features:", X_values.shape[1])
print("feature_names:", feature_names)
df

X shape: (3446, 27)
Number of input features: 27
feature_names: ['Th', 'C', 'Cr', 'Co', 'Mo', 'W', 'Al', 'Ti', 'Nb', 'B', 'Fe', 'Y', 'Zr', 'Ta', 'Re', 'Ru', 'V', 'La', 'S', 'Si', 'Mn', 'P', 'Hf', 'Cu', 'Ge', 'Ga', 'Ni']


,T.K,Th,C,Cr,Co,Mo,W,Al,Ti,Nb,...,La,S,Si,Mn,P,Hf,Cu,Ge,Ga,Ni
0,1144.2600,0.0,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.00,0.00,0.0,0.50,0.00,0.0,0.00,99.500
1,1144.2600,0.0,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.00,0.00,0.0,0.75,0.00,0.0,0.00,99.250
2,1144.2600,0.0,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.00,0.00,0.0,0.50,0.00,0.0,0.05,99.450
3,1144.2600,0.0,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.00,0.00,0.0,0.50,0.00,0.0,0.20,99.300
4,1144.2600,0.0,0.00,0.0,0.0,0.0,0.0,0.00,0.0,0.0,...,0.0,0.0,0.00,0.00,0.0,0.50,0.00,0.0,0.50,99.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3441,922.0389,0.0,0.02,17.0,28.4,3.4,1.9,1.03,3.1,1.1,...,0.0,6.0,0.05,0.05,0.0,0.00,0.02,0.0,0.00,33.904
3442,866.4833,0.0,0.02,17.0,28.4,3.4,1.9,1.03,3.1,1.1,...,0.0,6.0,0.05,0.05,0.0,0.00,0.02,0.0,0.00,33.904
3443,1199.8170,0.0,0.18,10.0,15.0,3.0,0.0,5.50,5.5,0.0,...,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,59.750
3444,1199.8170,0.0,0.18,10.0,15.0,3.0,0.0,5.50,5.5,0.0,...,0.0,0.0,0.00,0.00,0.0,0.00,0.00,0.0,0.00,59.750


In [62]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_temp, y_train, y_temp = train_test_split(
    X_values,
    y_values,
    test_size=0.30,
    random_state=42,
    shuffle=True
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    shuffle=True
)


In [63]:
scale = True

if scale:
    scaler_X = StandardScaler()
    scaler_y = StandardScaler()

    X_train_scaled = scaler_X.fit_transform(X_train).astype("float32")
    X_val_scaled = scaler_X.transform(X_val).astype("float32")
    X_test_scaled = scaler_X.transform(X_test).astype("float32")

    y_train_scaled = scaler_y.fit_transform(y_train).astype("float32")
    y_val_scaled = scaler_y.transform(y_val).astype("float32")
    y_test_scaled = scaler_y.transform(y_test).astype("float32")
else:
    X_train_scaled = X_train
    X_val_scaled   = X_val
    X_test_scaled  = X_test
    y_train_scaled = y_train
    y_val_scaled   = y_val
    y_test_scaled  = y_test


In [64]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

class AlloyDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


In [65]:
batch_size = 32

train_loader = DataLoader(
    AlloyDataset(X_train_scaled, y_train_scaled),
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    AlloyDataset(X_val_scaled, y_val_scaled),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    AlloyDataset(X_test_scaled, y_test_scaled),
    batch_size=batch_size,
    shuffle=False
)


In [66]:
def build_group_indices(groups, feature_names):
    group_indices = {}

    for group_id, elements in groups.items():
        indices = []

        for el in elements:
            col_name = f"{el}"

            if col_name in feature_names:
                indices.append(feature_names.index(col_name))

        if len(indices) > 0:
            group_indices[group_id] = indices

    return group_indices




In [67]:
group_indices = build_group_indices(GROUPS, feature_names)

for group_id, indices in group_indices.items():
    cols = [feature_names[i] for i in indices]
    print(group_id, cols)


1 ['Co', 'Cr', 'Fe', 'Mo', 'W', 'Ta', 'Re']
2 ['W', 'Ta', 'Ti', 'Mo', 'Nb', 'Hf']
3 ['Cr']
4 ['Cr', 'Mo', 'W']
5 ['Mo', 'W', 'Nb']
6 ['C']
7 ['Al', 'Ti']
8 ['Co']
9 ['Al', 'Ti', 'Nb']
10 ['Al', 'Cr', 'Y', 'La']
11 ['La', 'Th']
12 ['Cr', 'Co', 'Si']
13 ['B', 'Ta']
14 ['B']
15 ['B', 'C', 'Zr', 'Hf']
16 ['Re', 'Ru']


In [68]:
class GroupDifferentiatingBranch(nn.Module):
    def __init__(self, input_indices, hidden_dim):
        super().__init__()

        self.register_buffer(
            "input_indices",
            torch.tensor(input_indices, dtype=torch.long)
        )

        input_dim = len(input_indices)

        self.positive = nn.Linear(input_dim, hidden_dim)
        self.negative = nn.Linear(input_dim, hidden_dim)

        nn.init.constant_(self.positive.bias, 0.5)
        nn.init.constant_(self.negative.bias, -0.5)

        self.activation = nn.Tanh()

    def forward(self, x):
        # x shape: [batch_size, total_elements]
        x_group = x.index_select(dim=1, index=self.input_indices)

        # direct signal: сумма элементов в группе
        direct = x_group.sum(dim=1, keepdim=True)

        pos = self.activation(self.positive(x_group))
        neg = self.activation(self.negative(x_group))

        # differentiated signal
        diff = pos - neg

        # выход группы
        out = torch.cat(
            [
                direct,
                pos,
                neg,
                diff
            ],
            dim=1
        )

        return out


In [69]:
class DifferentiatingSuperalloyNet(nn.Module):
    def __init__(
        self,
        input_dim,
        group_indices,
        branch_hidden_dim=8,
        head_hidden_dim=32,
        dropout=0.0
    ):
        super().__init__()

        self.branches = nn.ModuleDict()

        for group_id, indices in group_indices.items():
            self.branches[str(group_id)] = GroupDifferentiatingBranch(
                input_indices=indices,
                hidden_dim=branch_hidden_dim
            )

        one_branch_out_dim = 1 + 3 * branch_hidden_dim
        total_branch_out_dim = len(self.branches) * one_branch_out_dim

        self.head = nn.Sequential(
            nn.Linear(total_branch_out_dim, head_hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden_dim, head_hidden_dim),
            nn.Tanh(),
            nn.Linear(head_hidden_dim, 1)
        )

    def forward(self, x):
        branch_outputs = []

        for branch in self.branches.values():
            branch_outputs.append(branch(x))

        x = torch.cat(branch_outputs, dim=1)

        out = self.head(x)

        return out


In [70]:
class DifferentiatingSuperalloyNetV2(nn.Module):
    def __init__(
        self,
        input_dim,
        group_indices,
        branch_hidden_dim=8,
        head_hidden_dim=64,
        dropout=0.0
    ):
        super().__init__()

        if len(group_indices) == 0:
            raise ValueError(
                "group_indices is empty. "
                "Check feature_names and element column names."
            )

        self.branches = nn.ModuleDict()

        for group_id, indices in group_indices.items():
            self.branches[str(group_id)] = GroupDifferentiatingBranch(
                input_indices=indices,
                hidden_dim=branch_hidden_dim
            )

        one_branch_out_dim = 1 + 3 * branch_hidden_dim
        total_branch_out_dim = len(self.branches) * one_branch_out_dim

        # Важно: добавляем input_dim, потому что будем передавать исходный x напрямую
        head_input_dim = total_branch_out_dim + input_dim

        self.head = nn.Sequential(
            nn.Linear(head_input_dim, head_hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),

            nn.Linear(head_hidden_dim, head_hidden_dim),
            nn.Tanh(),

            nn.Linear(head_hidden_dim, 1)
        )

    def forward(self, x):
        branch_outputs = []

        for branch in self.branches.values():
            branch_outputs.append(branch(x))

        grouped_signal = torch.cat(branch_outputs, dim=1)

        # direct + differentiating signal
        combined = torch.cat([x, grouped_signal], dim=1)

        out = self.head(combined)

        return out


In [71]:
def train_one_model(
    model,
    train_loader,
    val_loader,
    lr=1e-3,
    weight_decay=1e-5,
    max_epochs=1000,
    patience=80,
    device="cpu"
):
    model = model.to(device)

    criterion = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_val_loss = float("inf")
    best_state = None
    patience_counter = 0

    history = {
        "train_loss": [],
        "val_loss": []
    }

    for epoch in tqdm(range(max_epochs), desc='training', leave=False):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            pred = model(xb)
            loss = criterion(pred, yb)

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_loss = np.mean(train_losses)

        model.eval()
        val_losses = []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)

                pred = model(xb)
                loss = criterion(pred, yb)

                val_losses.append(loss.item())

        val_loss = np.mean(val_losses)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            break

    model.load_state_dict(best_state)

    return model, best_val_loss, history


In [72]:
def predict_scaled(model, loader, device="cpu"):
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)

            pred = model(xb).cpu().numpy()

            preds.append(pred)
            targets.append(yb.numpy())

    preds = np.vstack(preds)
    targets = np.vstack(targets)

    return preds, targets

def inverse_y(y_scaled, scaler_y):
    return scaler_y.inverse_transform(y_scaled)


In [73]:
def compute_metrics(y_pred, y_true):
    y_pred = y_pred.reshape(-1)
    y_true = y_true.reshape(-1)

    mae = np.mean(np.abs(y_pred - y_true))
    rmse = np.sqrt(np.mean((y_pred - y_true) ** 2))

    rmae_percent = np.mean(np.abs((y_pred - y_true) / y_true)) * 100

    rrmse = rmse / np.mean(y_true)

    return {
        "MAE": mae,
        "RMSE": rmse,
        "RMAE_percent": rmae_percent,
        "RRMSE": rrmse
    }


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

input_dim = X_train_scaled.shape[1]

results = []

for branch_hidden_dim in tqdm(range(4, 21), desc="Hidden dim search"):
    torch.manual_seed(42)
    np.random.seed(42)

    model = DifferentiatingSuperalloyNetV2(
        input_dim=input_dim,
        group_indices=group_indices,
        branch_hidden_dim=branch_hidden_dim,
        head_hidden_dim=32,
        dropout=0.0
    )

    model, best_val_loss, history = train_one_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        lr=1e-3,
        weight_decay=1e-5,
        max_epochs=10,
        patience=80,
        device=device
    )

    y_val_pred_scaled, y_val_true_scaled = predict_scaled(
        model,
        val_loader,
        device=device
    )

    if scale:
        y_val_pred = inverse_y(y_val_pred_scaled, scaler_y)
        y_val_true = inverse_y(y_val_true_scaled, scaler_y)
    else:
        y_val_pred = y_val_pred_scaled
        y_val_true = y_val_true_scaled

    val_metrics = compute_metrics(y_val_pred, y_val_true)

    result = {
        "branch_hidden_dim": branch_hidden_dim,
        "best_val_loss_scaled": best_val_loss,
        **val_metrics
    }

    results.append(result)

    print(
        f"branch_hidden={branch_hidden_dim:2d} | "
        f"MAE={val_metrics['MAE']:.3f} | "
        f"RMSE={val_metrics['RMSE']:.3f} | "
        f"RMAE={val_metrics['RMAE_percent']:.2f}% | "
        f"RRMSE={val_metrics['RRMSE']:.4f}"
    )


Device: cpu


Hidden dim search:   6%|▌         | 1/17 [00:09<02:30,  9.38s/it]

branch_hidden= 4 | MAE=47.153 | RMSE=71.591 | RMAE=4.10% | RRMSE=0.0607


Hidden dim search:  12%|█▏        | 2/17 [00:19<02:25,  9.73s/it]

branch_hidden= 5 | MAE=48.620 | RMSE=74.458 | RMAE=4.26% | RRMSE=0.0631


...

branch_hidden=18 | MAE=46.150 | RMSE=69.129 | RMAE=3.92% | RRMSE=0.0586


Hidden dim search:  94%|█████████▍| 16/17 [02:17<00:06,  6.68s/it]

branch_hidden=19 | MAE=40.727 | RMSE=65.992 | RMAE=3.49% | RRMSE=0.0560


Hidden dim search: 100%|██████████| 17/17 [02:24<00:00,  8.48s/it]

branch_hidden=20 | MAE=43.841 | RMSE=68.586 | RMAE=3.73% | RRMSE=0.0582


In [75]:
if scale:
        y_train_orig = inverse_y(y_train_scaled, scaler_y)
        y_val_orig = inverse_y(y_val_scaled, scaler_y)
else:
        y_train_orig = y_train_scaled
        y_val_orig = y_val_scaled


baseline_pred = np.full_like(
    y_val_orig,
    fill_value=y_train_orig.mean()
)

baseline_metrics = compute_metrics(
    baseline_pred,
    y_val_orig
)

baseline_metrics


{'MAE': np.float32(120.06752),
 'RMSE': np.float32(142.69817),
 'RMAE_percent': np.float32(10.470457),
 'RRMSE': np.float32(0.12100632)}

In [76]:
print("X_train_scaled finite:", np.isfinite(X_train_scaled).all())
print("X_val_scaled finite:", np.isfinite(X_val_scaled).all())
print("y_train_scaled finite:", np.isfinite(y_train_scaled).all())
print("y_val_scaled finite:", np.isfinite(y_val_scaled).all())

print("X_train_scaled mean:", X_train_scaled.mean())
print("X_train_scaled std:", X_train_scaled.std())

print("y_train_scaled mean:", y_train_scaled.mean())
print("y_train_scaled std:", y_train_scaled.std())


X_train_scaled finite: True
X_val_scaled finite: True
y_train_scaled finite: True
y_val_scaled finite: True
X_train_scaled mean: -1.2535244e-08
X_train_scaled std: 1.0
y_train_scaled mean: 3.7403643e-07
y_train_scaled std: 0.99999994


In [77]:
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("RMAE_percent")

results_df

,branch_hidden_dim,best_val_loss_scaled,MAE,RMSE,RMAE_percent,RRMSE
15,19,0.275929,40.727230,65.992264,3.488974,0.055961
7,11,0.285884,41.616180,67.843307,3.572262,0.057530
13,17,0.296675,42.480507,69.260475,3.620735,0.058732
11,15,0.272249,43.069374,68.603798,3.696630,0.058175
5,9,0.290502,43.605743,68.824265,3.721979,0.058362
16,20,0.297500,43.840607,68.585503,3.730319,0.058160
2,6,0.313765,44.748302,70.214561,3.766768,0.059541
10,14,0.308951,44.476757,69.932442,3.809515,0.059302
6,10,0.307094,45.634602,70.886604,3.869541,0.060111
9,13,0.309921,44.916798,70.429771,3.907898,0.059724


In [78]:
best_branch_hidden_dim = int(results_df.iloc[0]["branch_hidden_dim"])

print("Best branch_hidden_dim:", best_branch_hidden_dim)


Best branch_hidden_dim: 19


In [79]:
torch.manual_seed(42)
np.random.seed(42)

best_model = DifferentiatingSuperalloyNetV2(
    input_dim=input_dim,
    group_indices=group_indices,
    branch_hidden_dim=best_branch_hidden_dim,
    head_hidden_dim=32,
    dropout=0.0
)

best_model, best_val_loss, best_history = train_one_model(
    model=best_model,
    train_loader=train_loader,
    val_loader=val_loader,
    lr=1e-3,
    weight_decay=1e-5,
    max_epochs=100,
    patience=80,
    device=device
)


In [80]:
y_test_pred_scaled, y_test_true_scaled = predict_scaled(
    best_model,
    test_loader,
    device=device
)

y_test_pred = inverse_y(y_test_pred_scaled, scaler_y)
y_test_true = inverse_y(y_test_true_scaled, scaler_y)

test_metrics = compute_metrics(y_test_pred, y_test_true)

test_metrics


{'MAE': np.float32(31.219023),
 'RMSE': np.float32(58.57935),
 'RMAE_percent': np.float32(2.6911395),
 'RRMSE': np.float32(0.050129645)}

In [81]:
pred_df = pd.DataFrame({
    "y_true": y_test_true.reshape(-1),
    "y_pred": y_test_pred.reshape(-1),
    "abs_error": np.abs(y_test_pred.reshape(-1) - y_test_true.reshape(-1)),
    "rel_error_percent": np.abs(
        (y_test_pred.reshape(-1) - y_test_true.reshape(-1)) 
        / y_test_true.reshape(-1)
    ) * 100
})

pred_df.head()


,y_true,y_pred,abs_error,rel_error_percent
0,1023.150024,1026.407837,3.257812,0.318410
1,1473.150024,1450.179810,22.970215,1.559258
2,1073.150024,1060.237427,12.912598,1.203243
3,1183.150024,1180.237549,2.912476,0.246163
4,1033.150024,1029.084351,4.065674,0.393522


In [82]:
def build_relationship_matrix(groups, feature_names):
    element_names = element

    R = np.zeros((len(groups), len(feature_names)), dtype=np.float32)

    group_ids = list(groups.keys())

    for i, group_id in enumerate(group_ids):
        for el in groups[group_id]:
            if el in element_names:
                j = element_names.index(el)
                R[i, j] = 1.0

    return R, group_ids, element_names

R, group_ids, element_names = build_relationship_matrix(
    GROUPS,
    feature_names
)

print(R.shape)


(16, 27)
